# Preprocessing on High Volume For-Hire Vehicle (HVFHV) Trip Records Dataset:

In this notebook, we are mainly focusing on removing outliers in HVFHV dataset and drop unused columns.

----

# Import Libraries:

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F 
from pyspark.sql.functions import min as spark_min, max as spark_max, col
from pyspark.sql.functions import * 
import os

In [ ]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("preprocessing_hvfhv")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.executor.heartbeatInterval", "30s")
    .config("spark.network.timeout", "600s")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.rpc.askTimeout", "600s")
    .config("spark.driver.memory", "100G")
    .config("spark.executor.memory", "100G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)

# Read HVFHV Parquet Files:

In [ ]:
base_dir = "../data"
hvfhv_path = base_dir + '/raw/hvfhv_data/'
hvfhv_sdf = spark.read.parquet(hvfhv_path)

In [ ]:
hvfhv_sdf.printSchema()

In [ ]:
# Check the shape of parquet file
num_rows = hvfhv_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hvfhv_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

In [ ]:
hvfhv_sdf.show(5)

Change the unit of `trip_time` from seconds to minutes:

In [ ]:
hvfhv_sdf = hvfhv_sdf.withColumn('trip_time', F.round(F.col('trip_time') / 60, 2))
hvfhv_sdf.show(5)

# Drop Unrelated Rows and Columns:

In [ ]:
# Calculate the amount of NULL in each column
na_counts = hvfhv_sdf.select([sum(col(column).isNull().cast("int")).alias(column) for column in hvfhv_sdf.columns])
na_counts.show()

Since imputing values in `originating_base_num` and `on_scene_datetime` are illogical, we drop these two columns.

In [ ]:
hvfhv_sdf = hvfhv_sdf.drop("originating_base_num", "on_scene_datetime")
hvfhv_sdf.show(5)

In [ ]:
num_rows = hvfhv_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hvfhv_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

In [ ]:
numeric_columns = ['trip_miles', 'trip_time', 'base_passenger_fare', 'tolls', 'bcf', 'sales_tax',
                  'congestion_surcharge', 'airport_fee', 'tips', 'driver_pay']

# Outlier Detection:

### Find the earliest and latest of datetime variables:

In [ ]:
# Time period for `request_datetime`
date_range = hvfhv_sdf.select(
    spark_min(col("request_datetime")).alias("min_date"),
    spark_max(col("request_datetime")).alias("max_date")
    ).collect()

print("request_datetime:")
print(f"Min: {date_range[0]['min_date']}")
print(f"Max: {date_range[0]['max_date']}")

# Time period for `pickup_datetime`
date_range = hvfhv_sdf.select(
    spark_min(col("pickup_datetime")).alias("min_date"),
    spark_max(col("pickup_datetime")).alias("max_date")
    ).collect()

print("pickup_datetime:")
print(f"Min: {date_range[0]['min_date']}")
print(f"Max: {date_range[0]['max_date']}")

# Time period for `dropoff_datetime`
date_range = hvfhv_sdf.select(
    spark_min(col("dropoff_datetime")).alias("min_date"),
    spark_max(col("dropoff_datetime")).alias("max_date")
    ).collect()

print("dropoff_datetime:")
print(f"Min: {date_range[0]['min_date']}")
print(f"Max: {date_range[0]['max_date']}")

Remove the rows when `request_datetime` starts with "2023-06-30" and "2024-01-01" and `dropoff_datetime` starts with "2024-01-01":

In [ ]:
hvfhv_sdf = hvfhv_sdf.filter(~(F.col('request_datetime').startswith('2023-06-30')) &
                              ~ (F.col('request_datetime').startswith('2024-01-01')) & 
                              ~(F.col('dropoff_datetime').startswith('2024-01-01')))

### Calculate the descriptive statistics of numerical columns:

In [ ]:
hvfhv_sdf.select(numeric_columns).describe().show()

### Filter out the unrealistic data:
- Remove the data in `PULocationID` and `DOLocationID` which out of the range
- Assume the `trip_miles` is greater than 1 mile, `trip_time` is greater than 1 minute, `base_passenger_fare` is greater than 1 dollar
- Assume the maximum `trip_time` is 300 minutes
- Assume `driver_pay` is greater than 0

In [ ]:
hvfhv_sdf = hvfhv_sdf.filter((hvfhv_sdf.PULocationID >= 1) & 
                             (hvfhv_sdf.PULocationID <= 263) &
                             (hvfhv_sdf.DOLocationID >= 1) & 
                             (hvfhv_sdf.DOLocationID <= 263) &
                             (hvfhv_sdf.trip_miles > 1) &
                             (hvfhv_sdf.trip_time > 1) & 
                             (hvfhv_sdf.trip_time <= 300) &
                             (hvfhv_sdf.base_passenger_fare > 1) &
                             (hvfhv_sdf.driver_pay > 0))

num_rows = hvfhv_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hvfhv_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

### Calculate the descriptive statistics of numerical columns again:

In [ ]:
hvfhv_sdf.select(numeric_columns).describe().show()

### Check the time period of the dataset again:

In [ ]:
# Time period for `request_datetime`
date_range = hvfhv_sdf.select(
    spark_min(col("request_datetime")).alias("min_date"),
    spark_max(col("request_datetime")).alias("max_date")
    ).collect()

print("request_datetime:")
print(f"Min: {date_range[0]['min_date']}")
print(f"Max: {date_range[0]['max_date']}")

# Time period for `pickup_datetime`
date_range = hvfhv_sdf.select(
    spark_min(col("pickup_datetime")).alias("min_date"),
    spark_max(col("pickup_datetime")).alias("max_date")
    ).collect()

print("pickup_datetime:")
print(f"Min: {date_range[0]['min_date']}")
print(f"Max: {date_range[0]['max_date']}")

# Time period for `dropoff_datetime`
date_range = hvfhv_sdf.select(
    spark_min(col("dropoff_datetime")).alias("min_date"),
    spark_max(col("dropoff_datetime")).alias("max_date")
    ).collect()

print("dropoff_datetime:")
print(f"Min: {date_range[0]['min_date']}")
print(f"Max: {date_range[0]['max_date']}")

Now, the HVFHV dataset is in the correct time period and its outliers have been removed.

# Save the Preprocessed HVFHV Dataset:

In [ ]:
hvfhv_dir = base_dir + '/curated/hvfhv_data'
file_name = 'preprocessed_hvfhv'
hvfhv_path = os.path.join(hvfhv_dir, file_name)
hvfhv_sdf.write.mode('overwrite').parquet(hvfhv_path)